In [1]:
import os
import re
import time
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
import psycopg2
from pgvector.psycopg2 import register_vector


ModuleNotFoundError: No module named 'pgvector'

In [3]:
# Config
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

PREPROCESSED_DIR = Path("../Preprocessed_Data")
EMBEDDING_MODEL = "gemini-embedding-001"

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "database": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD")
}

def get_connection():
    conn = psycopg2.connect(**DB_CONFIG)
    register_vector(conn)
    return conn

In [17]:
def setup_database():
    create_table_sql = '''
    CREATE TABLE IF NOT EXISTS chunks (
        id SERIAL PRIMARY KEY,
        chunk_id TEXT UNIQUE NOT NULL,
        parent_doc_id TEXT NOT NULL,
        source TEXT,
        category TEXT,
        year TEXT,
        chunk_index INTEGER,
        chunk_title TEXT,
        topic_tags TEXT,
        summary TEXT,
        content TEXT NOT NULL,
        embedding vector(3072)
    );
    '''
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(create_table_sql)
        conn.commit()
    print("Database and index ready.")

In [5]:
def parse_preprocessed_file(file_path):
    chunks = []
    with open(file_path, encoding="utf-8") as f:
        content = f.read()
    chunk_blocks = re.split(r"<<<CHUNK_START>>>|<<<CHUNK_END>>>", content)
    for block in chunk_blocks:
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        meta = {}
        i = 0
        while i < len(lines) and lines[i].strip():
            if ':' in lines[i]:
                k, v = lines[i].split(':', 1)
                meta[k.strip()] = v.strip()
            i += 1
        while i < len(lines) and not lines[i].strip():
            i += 1
        content_text = '\n'.join(lines[i:]).strip()
        meta['content'] = content_text
        if meta.get('chunk_id') and meta.get('content'):
            chunks.append(meta)
    return chunks

In [20]:
def embed_chunk(chunk, max_retries=3):
    text = (chunk.get('summary', '') + "\n\n" + chunk.get('content', '')).strip()
    for attempt in range(max_retries):
        try:
            response = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=text,
                config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
            )
            return response.embeddings[0].values
        except Exception as e:
            msg = str(e)
            if any(code in msg for code in ["429", "503"]):
                print(f"Gemini rate limit, retry {attempt+1}/3...")
                time.sleep(30)
            else:
                print(f"Embedding error: {e}")
                break
    return None

In [6]:
def store_chunk(cur, chunk, embedding):
    sql = '''
    INSERT INTO chunks (chunk_id, parent_doc_id, source, category, year,
                        chunk_index, chunk_title, topic_tags, summary, content, embedding)
    VALUES (%(chunk_id)s, %(parent_doc_id)s, %(source)s, %(category)s, %(year)s,
            %(chunk_index)s, %(chunk_title)s, %(topic_tags)s, %(summary)s, %(content)s, %(embedding)s)
    ON CONFLICT (chunk_id) DO NOTHING;
    '''
    params = {
        'chunk_id': chunk.get('chunk_id'),
        'parent_doc_id': chunk.get('parent_doc_id'),
        'source': chunk.get('source'),
        'category': chunk.get('category'),
        'year': chunk.get('year'),
        'chunk_index': int(chunk.get('chunk_index')) if chunk.get('chunk_index') else None,
        'chunk_title': chunk.get('chunk_title'),
        'topic_tags': chunk.get('topic_tags'),
        'summary': chunk.get('summary'),
        'content': chunk.get('content'),
        'embedding': embedding
    }
    cur.execute(sql, params)

In [7]:
def embed_and_store_file(file_path):
    chunks = parse_preprocessed_file(file_path)
    with get_connection() as conn:
        with conn.cursor() as cur:
            for chunk in chunks:
                try:
                    cur.execute("SELECT 1 FROM chunks WHERE chunk_id=%s", (chunk.get('chunk_id'),))
                    if cur.fetchone():
                        print(f"[SKIP] {chunk.get('chunk_id')}")
                        continue
                    embedding = embed_chunk(chunk)
                    if embedding is None:
                        print(f"[FAIL] {chunk.get('chunk_id')} (embedding error)")
                        continue
                    store_chunk(cur, chunk, embedding)
                    conn.commit()
                    print(f"[OK] {chunk.get('chunk_id')}")
                    time.sleep(0.5)
                except Exception as e:
                    conn.rollback()
                    print(f"[FAIL] {chunk.get('chunk_id')} ({e})")

In [8]:
def process_single_file(path_str):
    file_path = Path(path_str)
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return
    print(f"Processing file: {file_path}")
    embed_and_store_file(file_path)

In [9]:
def process_category(category_name):
    cat_dir = PREPROCESSED_DIR / category_name
    if not cat_dir.exists():
        print(f"Category not found: {cat_dir}")
        return
    for file_path in cat_dir.rglob("*_preprocessed.txt"):
        if "_review" in file_path.parts:
            continue
        process_single_file(str(file_path))

In [10]:
def process_all():
    for category_dir in PREPROCESSED_DIR.iterdir():
        if not category_dir.is_dir() or category_dir.name == "_review":
            continue
        for file_path in category_dir.rglob("*_preprocessed.txt"):
            if "_review" in file_path.parts:
                continue
            process_single_file(str(file_path))

In [21]:
setup_database()

Database and index ready.


In [22]:
process_category("Hoc_phi")

Processing file: ..\Preprocessed_Data\Hoc_phi\Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH_preprocessed.txt
[OK] Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH_001
[OK] Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH_002
[OK] Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH_003
[OK] Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH_004
[OK] Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH_005
[OK] Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH_006
[OK] Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH_007
[OK] Hoc_phi_2025_DHCQ_KSCS_VLVH_SDH_008


In [ ]:
def retrieve(query: str, top_k: int = 5, category: str = None) -> list[dict]:
    """
    Retrieve top_k most similar chunks from PostgreSQL using dense vector search.
    
    Args:
        query: Query text to embed and search
        top_k: Number of top results to return
        category: Optional category filter
    
    Returns:
        List of dictionaries with chunk metadata and similarity score
    """
    # Embed query
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
    )
    query_embedding = response.embeddings[0].values
    
    # Build SQL query
    base_sql = '''
    SELECT chunk_id, parent_doc_id, category, chunk_title, summary, content,
           1 - (embedding <=> %s::vector) AS similarity
    FROM chunks
    '''
    
    if category:
        sql = base_sql + f"WHERE category = %s "
        params = [query_embedding, category]
    else:
        sql = base_sql
        params = [query_embedding]
    
    sql += "ORDER BY embedding <=> %s::vector LIMIT %s"
    params.extend([query_embedding, top_k])
    
    # Execute query
    results = []
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            rows = cur.fetchall()
            
            for row in rows:
                results.append({
                    'chunk_id': row[0],
                    'parent_doc_id': row[1],
                    'category': row[2],
                    'chunk_title': row[3],
                    'summary': row[4],
                    'content': row[5],
                    'similarity': row[6]
                })
    
    return results


def pretty_print(results):
    """
    Print retrieval results in a formatted, readable way.
    
    Args:
        results: List of result dictionaries from retrieve()
    """
    for idx, result in enumerate(results, 1):
        print(f"[{idx}] {result['chunk_id']} | similarity: {result['similarity']:.2f}")
        if result['chunk_title']:
            print(f"    Title: {result['chunk_title']}")
        if result['summary']:
            print(f"    Summary: {result['summary'][:200]}...")
        print("    ---")